# Garbage Classification Image Collector

YOLO 学習用にゴミ分別画像を収集する Colab ノートブック。

**カテゴリ**
1. `plastic_container` — プラ容器 (カップ・トレイ・袋)
2. `paper_cardboard` — 紙・段ボール
3. `pet_bottle` — ペットボトル

**フロー**
1. `icrawler` で Google / Bing から各キーワードを検索 → `_raw/` に保存 (ハッシュ命名で自動重複除外)
2. CLIP (ViT-B/32) で「正解カテゴリ / 他カテゴリ / その他 (ノイズ)」の4値ゼロショット分類 → `scores.csv`
3. 正解が最高スコアのものから上位 50 枚を `{category}_001.jpg` 形式で各カテゴリフォルダに保存

**再開可能性**
- 各カテゴリの収集セルは独立実行可
- 既存の `_raw/` ファイルはハッシュで重複検出するので再実行しても増えるだけ
- `scores.csv` を作り直せば選別だけやり直せる (再DL不要)

**Drive 出力**
```
MyDrive/garbage_dataset/
├── plastic_container/       50枚
├── paper_cardboard/         50枚
├── pet_bottle/              50枚
├── _raw/{category}/         CLIP 前の全収集画像
└── scores.csv               CLIP スコア
```


## 1. セットアップ

In [ ]:
!pip install -q icrawler transformers


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/garbage_dataset')
RAW_DIR = DRIVE_ROOT / '_raw'
SCORES_CSV = DRIVE_ROOT / 'scores.csv'

CATEGORIES = {
    'plastic_container': {
        'keywords': [
            'プラスチック容器 ゴミ',
            '使用済み プラカップ',
            '食品トレイ 発泡スチロール',
            'レジ袋 プラスチック',
            'プラごみ 分別',
        ],
        'clip_prompt': 'a photo of a used plastic container, cup, tray, or plastic bag',
    },
    'paper_cardboard': {
        'keywords': [
            '段ボール ゴミ',
            '古紙 回収',
            '新聞紙 束',
            '紙くず ゴミ',
            'ダンボール箱 つぶす',
        ],
        'clip_prompt': 'a photo of waste paper, newspapers, or cardboard boxes',
    },
    'pet_bottle': {
        'keywords': [
            'ペットボトル ゴミ',
            '空のペットボトル',
            'ペットボトル 回収',
            'PETボトル 分別',
            'ペットボトル つぶす',
        ],
        'clip_prompt': 'a photo of an empty PET plastic bottle',
    },
}

OTHER_PROMPT = 'a photo of something else, an illustration, a logo, or a person'

PER_KEYWORD = 25
FINAL_PER_CATEGORY = 50
MIN_IMAGE_SIZE = (200, 200)

CATEGORY_ORDER = list(CATEGORIES.keys())

for c in CATEGORY_ORDER:
    (RAW_DIR / c).mkdir(parents=True, exist_ok=True)
    (DRIVE_ROOT / c).mkdir(parents=True, exist_ok=True)

print('Drive root:', DRIVE_ROOT)
print('Categories:', CATEGORY_ORDER)


## 2. 画像収集 (icrawler)

各カテゴリのセルは**独立して実行可能**。途中で Colab が切れても、もう一度同じセルを実行すれば既存ハッシュをスキップして続きから収集します。

`_raw/{category}/` には `{hash}.jpg` 形式で保存 (重複除外用)。最終リネームは CLIP 選別後に行います。


In [ ]:
import hashlib
import shutil
from icrawler.builtin import GoogleImageCrawler, BingImageCrawler

def file_hash(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

def collect_category(category, engines=('bing', 'google')):
    cat_raw = RAW_DIR / category
    cat_raw.mkdir(parents=True, exist_ok=True)

    before = len(list(cat_raw.glob('*.*')))
    print(f'[{category}] _raw existing: {before}')

    keywords = CATEGORIES[category]['keywords']
    tmp_root = Path(f'/content/tmp_{category}')
    if tmp_root.exists():
        shutil.rmtree(tmp_root)

    for i, kw in enumerate(keywords):
        for engine in engines:
            tmp_dir = tmp_root / f'{i:02d}_{engine}'
            tmp_dir.mkdir(parents=True, exist_ok=True)

            crawler_cls = GoogleImageCrawler if engine == 'google' else BingImageCrawler
            crawler = crawler_cls(
                storage={'root_dir': str(tmp_dir)},
                feeder_threads=1,
                parser_threads=1,
                downloader_threads=4,
            )
            print(f"  [{engine}] '{kw}'")
            try:
                crawler.crawl(
                    keyword=kw,
                    max_num=PER_KEYWORD,
                    min_size=MIN_IMAGE_SIZE,
                    file_idx_offset=0,
                )
            except Exception as e:
                print(f'    failed: {e}')
                continue

            for src in tmp_dir.glob('*.*'):
                if src.suffix.lower() not in {'.jpg', '.jpeg', '.png'}:
                    continue
                try:
                    h = file_hash(src)
                except Exception:
                    continue
                ext = '.jpg' if src.suffix.lower() == '.jpeg' else src.suffix.lower()
                dst = cat_raw / f'{h[:16]}{ext}'
                if not dst.exists():
                    shutil.copy2(src, dst)

    shutil.rmtree(tmp_root, ignore_errors=True)
    after = len(list(cat_raw.glob('*.*')))
    print(f'[{category}] _raw total: {after} (+{after - before})')


### 2-1. プラ容器

In [ ]:
collect_category('plastic_container')


### 2-2. 紙・段ボール

In [ ]:
collect_category('paper_cardboard')


### 2-3. ペットボトル

In [ ]:
collect_category('pet_bottle')


## 3. CLIP ゼロショット分類

各画像を以下 4 つのプロンプトとの類似度で評価し、正解カテゴリが最高スコアのものだけ採用します。

- `plastic_container`: *a photo of a used plastic container, cup, tray, or plastic bag*
- `paper_cardboard`: *a photo of waste paper, newspapers, or cardboard boxes*
- `pet_bottle`: *a photo of an empty PET plastic bottle*
- `other` (ノイズ除外): *a photo of something else, an illustration, a logo, or a person*


In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

clip_model = CLIPModel.from_pretrained('openai/clip-vit-base-patch32').to(device).eval()
clip_processor = CLIPProcessor.from_pretrained('openai/clip-vit-base-patch32')

PROMPTS = [CATEGORIES[c]['clip_prompt'] for c in CATEGORY_ORDER] + [OTHER_PROMPT]
for p in PROMPTS:
    print('  -', p)


In [ ]:
import csv
from PIL import Image, UnidentifiedImageError
from tqdm import tqdm

@torch.no_grad()
def encode_text(prompts):
    inputs = clip_processor(text=prompts, return_tensors='pt', padding=True).to(device)
    feats = clip_model.get_text_features(**inputs)
    return feats / feats.norm(dim=-1, keepdim=True)

@torch.no_grad()
def encode_images(pil_images):
    inputs = clip_processor(images=pil_images, return_tensors='pt').to(device)
    feats = clip_model.get_image_features(**inputs)
    return feats / feats.norm(dim=-1, keepdim=True)

def load_image(p):
    try:
        img = Image.open(p).convert('RGB')
        if min(img.size) < MIN_IMAGE_SIZE[0]:
            return None
        return img
    except (UnidentifiedImageError, OSError):
        return None

text_feats = encode_text(PROMPTS)

rows = []
BATCH = 32
for cat in CATEGORY_ORDER:
    cat_raw = RAW_DIR / cat
    paths = sorted(p for p in cat_raw.glob('*.*') if p.is_file())
    print(f'[{cat}] scoring {len(paths)} image(s)')
    for i in tqdm(range(0, len(paths), BATCH)):
        chunk = paths[i:i + BATCH]
        imgs, kept = [], []
        for p in chunk:
            img = load_image(p)
            if img is not None:
                imgs.append(img)
                kept.append(p)
        if not imgs:
            continue
        img_feats = encode_images(imgs)
        sims = (img_feats @ text_feats.T).cpu().numpy()
        for p, s in zip(kept, sims):
            rows.append([cat, p.name, float(s[0]), float(s[1]), float(s[2]), float(s[3])])

with open(SCORES_CSV, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['raw_category', 'filename',
                'score_plastic', 'score_paper', 'score_pet', 'score_other'])
    w.writerows(rows)

print(f'scores saved: {SCORES_CSV} ({len(rows)} rows)')


## 4. 上位 50 枚を選別して最終フォルダへ

`scores.csv` から各カテゴリの上位を選んで `{category}/{category}_NNN.jpg` に連番コピー。

**この手順は何度でもやり直せます** — `_raw/` と `scores.csv` を保持しているので、閾値や枚数を変えたければこのセルだけ再実行すればOK。


In [ ]:
import pandas as pd

df = pd.read_csv(SCORES_CSV)
score_cols = ['score_plastic', 'score_paper', 'score_pet', 'score_other']
col_to_cat = {
    'score_plastic': 'plastic_container',
    'score_paper': 'paper_cardboard',
    'score_pet': 'pet_bottle',
    'score_other': 'other',
}
df['pred'] = df[score_cols].idxmax(axis=1).map(col_to_cat)

cat_to_col = {
    'plastic_container': 'score_plastic',
    'paper_cardboard': 'score_paper',
    'pet_bottle': 'score_pet',
}

for cat, col in cat_to_col.items():
    final_dir = DRIVE_ROOT / cat
    for p in final_dir.glob('*.*'):
        p.unlink()

    sub = df[(df['raw_category'] == cat) & (df['pred'] == cat)].copy()
    sub = sub.sort_values(col, ascending=False).head(FINAL_PER_CATEGORY)
    print(f'[{cat}] candidates after CLIP: {len(df[(df.raw_category==cat) & (df.pred==cat)])}, '
          f'selected top: {len(sub)} / target {FINAL_PER_CATEGORY}')

    for i, row in enumerate(sub.itertuples(index=False), start=1):
        src = RAW_DIR / cat / row.filename
        ext = Path(row.filename).suffix.lower()
        dst = final_dir / f'{cat}_{i:03d}{ext}'
        shutil.copy2(src, dst)

    if len(sub) < FINAL_PER_CATEGORY:
        print(f"  ⚠ under target — rerun collect_category('{cat}') for more raw images.")


## 5. 結果プレビュー

In [ ]:
import math
import matplotlib.pyplot as plt

def show_grid(cat, n_cols=10):
    final_dir = DRIVE_ROOT / cat
    paths = sorted(final_dir.glob('*.*'))
    if not paths:
        print(f'[{cat}] no images')
        return
    n_rows = math.ceil(len(paths) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.6, n_rows * 1.6))
    fig.suptitle(f'{cat}  ({len(paths)} images)', fontsize=14)
    axes_flat = axes.flat if hasattr(axes, 'flat') else [axes]
    for ax, p in zip(axes_flat, paths):
        try:
            ax.imshow(Image.open(p))
        except Exception:
            pass
        ax.axis('off')
        ax.set_title(p.stem.split('_')[-1], fontsize=6)
    for ax in list(axes_flat)[len(paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

for cat in CATEGORY_ORDER:
    show_grid(cat)


## 6. トラブルシュート

### 50 枚に届かない場合
該当カテゴリの `collect_category(...)` セルをもう一度実行 → その後セクション 3, 4 を再実行。

### ノイズが多い/精度を上げたい
- `CATEGORIES[...]['clip_prompt']` を編集して再実行 (セクション 3, 4 のみ)
- `OTHER_PROMPT` を強化 (例: `'a cartoon illustration or a product advertisement'`)
- `PER_KEYWORD` を増やして収集母数を増やす

### Colab がタイムアウトした
どのセルでも、もう一度実行すれば既存ファイル/スコアを活かして続きから再開します。
